# Imidazolone & Thiazolone Library Generation

Virtual combinatorial library of COX-2 selective inhibitors derived from  
commercially available aldehydes, carboxylic acids and primary amines via  
reactions *in silico*.

**Pipeline:** ```Aldehydes + Carboxylic Acids → Erlenmeyer-Plöchl → Oxazolones → Aminolysis-GFPc → Imidazolones```  

**Amines** feed into the Aminolysis-GFPc step as co-reactants.  

**Parallel branch:** ```Oxazolones → Sulphur-Exchange → Thiazolones```  

All three building-block sets pass Price → Veber filters before reactions.  
Final products (Imidazolones and Thiazolones) are filtered by Veber and  
Brenk+PAINS before export, and saved as two **separate** CSV files.


In [1]:
import pandas as pd

import py_utils as pu

pu.init_stage_dirs()

FORCE_RECOMPUTE = False

print(f"py_utils v{pu.__version__}")

py_utils v3.18


## 📥 1. Load SDFs

The SDFs of the compounds provided by Enamine (with their IDs) that match a
specific substructure are loaded:
- Aldehydes, R-CHO
- Carboxylic compounds, R-COOH or RCOX
- Primary amines, R-NH₂  
  
  
> SDF loading is fast — no caching needed.

In [ ]:
SDF_PATHS = { # Original SDF_PATHS for full dataset
    "aldehydes":   "mol_files/1. Enamine SDFs/EnamineBBStock_site_Aldehydes_12040.sdf",
    "carboxylics": "mol_files/1. Enamine SDFs/EnamineBBStock_site_CarboxylicAcids_75732.sdf",
    "amines":      "mol_files/1. Enamine SDFs/EnamineBBStock_site_PrimaryAmines_68264.sdf",
}

SDF_PATHS_test = { # SDF_PATHS_test for quick testing
    "aldehydes":   "mol_files/1. Enamine SDFs/TESTER_Aldehydes_120.sdf",
    "carboxylics": "mol_files/1. Enamine SDFs/TESTER_CarboxylicAcids_750.sdf",
    "amines":      "mol_files/1. Enamine SDFs/TESTER_PrimaryAmines_680.sdf",
}

df_aldehydes_raw   = pu.sdf_to_dataframe(SDF_PATHS["aldehydes"],   id_prefix="A") 
df_carboxylics_raw = pu.sdf_to_dataframe(SDF_PATHS["carboxylics"], id_prefix="C")
df_amines_raw      = pu.sdf_to_dataframe(SDF_PATHS["amines"],      id_prefix="N")

pu.report_df_size(df_aldehydes_raw,   "Aldehydes - Raw")
pu.report_df_size(df_carboxylics_raw, "Carboxylics - Raw")
pu.report_df_size(df_amines_raw,      "Amines - Raw")

[SDF Reader] Loaded 120 unique compounds from mol_files/1. Enamine SDFs/TESTER_Aldehydes_120.sdf
[SDF Reader] Loaded 750 unique compounds from mol_files/1. Enamine SDFs/TESTER_CarboxylicAcids_750.sdf
[SDF Reader] Loaded 680 unique compounds from mol_files/1. Enamine SDFs/TESTER_PrimaryAmines_680.sdf
[Aldehydes - Raw] 120 rows
[Carboxylics - Raw] 750 rows
[Amines - Raw] 680 rows


## 🔸 2. Price Filtration

Query Enamine Store API for real-time pricing. Compounds exceeding price thresholds are discarded during retrieval.

> Internal caching: prices are cached in `mol_files/2. Building Blocks/price_cache/`.

In [3]:
client = pu.EnamineClient()

print("=== Aldehydes ===")
df_aldehydes_cheap = pu.add_enamine_prices(df_aldehydes_raw, client=client)

print("\n=== Carboxylic Acids ===")
df_carboxylics_cheap = pu.add_enamine_prices(df_carboxylics_raw, client=client)

print("\n=== Amines ===")
df_amines_cheap = pu.add_enamine_prices(df_amines_raw, client=client)

pu.report_df_size(df_aldehydes_cheap,   "Aldehydes - Cheap")
pu.report_df_size(df_carboxylics_cheap, "Carboxylics - Cheap")
pu.report_df_size(df_amines_cheap,      "Amines - Cheap")

=== Aldehydes ===
[Enamine Pricing] Processing 120 compounds...
[Enamine Pricing] Querying 120 compounds via API
[Enamine Pricing] Filters: <= 40.0 EUR/g, <= 250.0 EUR/pack
[Enamine Pricing] Batch 1/1: 120 compounds
[EnamineClient] Signed in successfully.
[Enamine Pricing] Saved 36 valid + 84 invalid to cache
[Enamine Pricing] Completed: 36/120 with valid pricing
⚠️  Removed 84 compounds (no valid pricing)

=== Carboxylic Acids ===
[Enamine Pricing] Loaded 36 valid + 84 invalid cached
[Enamine Pricing] Processing 750 compounds...
[Enamine Pricing] Querying 750 compounds via API
[Enamine Pricing] Filters: <= 40.0 EUR/g, <= 250.0 EUR/pack
[Enamine Pricing] Batch 1/1: 750 compounds
[Enamine Pricing] Saved 182 valid + 688 invalid to cache
[Enamine Pricing] Completed: 146/750 with valid pricing
⚠️  Removed 604 compounds (no valid pricing)

=== Amines ===
[Enamine Pricing] Loaded 182 valid + 688 invalid cached
[Enamine Pricing] Processing 680 compounds...
[Enamine Pricing] Querying 680 compo

## 🔸 3. Veber filtering (of building blocks)

Compute RDKit molecular descriptors, then apply per-class bioavailability criteria.  

Each building block class uses independent limits (`VEBER_ALDEHYDES`, `VEBER_CARBOXYLICS`,`VEBER_AMINES`) back-calculated  
from the final product thresholds using each reaction's property increments (ΔtPSA, ΔRotB, ΔLogP, ΔMW, ΔHBD, ΔHBA).  

| Reaction | ΔtPSA | ΔRotB | ΔLogP | ΔMW | ΔHBD | ΔHBA |
|----------|------|-------|--------|------|------|------|
| E-P      | -15.71 | 0 | 0.4127 | +21.022 | -1 | +1 |
| A-G      | -32.01 | 1 | 0.3403 | -18.015 | -1 | -2 |
| S-E      | +16.07 | 0 | +0.7166 | +16.068 | -1 | 0 |

In [4]:
VEBER_BB = {
    "aldehydes":   dict(max_tPSA=118.41, max_RotB=10, max_LogP=4.4964, max_MWT=405.883, max_HBD=5, max_HBA=9),
    "carboxylics": dict(max_tPSA=138.64, max_RotB=10, max_LogP=4.3821, max_MWT=421.882, max_HBD=6, max_HBA=9),
    "amines":      dict(max_tPSA=133.35, max_RotB=9,  max_LogP=3.7943, max_MWT=377.873, max_HBD=6, max_HBA=9),
}

BB_ID_COLS = ["ID", "Catalog_ID", "SMILES", "PriceMol", "PriceG", "tPSA", "RotB", "LogP", "MW", "HBD", "HBA"]
BB_REJ_COLS = BB_ID_COLS + ["Rejection"]

def _veber_bb_filter(df: pd.DataFrame, veber_params: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    return pu.filter_Veber(pu.add_rdkit_properties(df), **veber_params)

df_aldehydes_druglike, _ = pu.load_or_filter(
    lambda: _veber_bb_filter(df_aldehydes_cheap, VEBER_BB["aldehydes"]),
    accepted_csv=pu.stage_path("Aldehydes"),
    rejected_csv=pu.rejected_path("Aldehydes"),
    force_recompute=FORCE_RECOMPUTE,
)

df_carboxylics_druglike, _ = pu.load_or_filter(
    lambda: _veber_bb_filter(df_carboxylics_cheap, VEBER_BB["carboxylics"]),
    accepted_csv=pu.stage_path("Carboxylics"),
    rejected_csv=pu.rejected_path("Carboxylics"),
    force_recompute=FORCE_RECOMPUTE,
)

df_amines_druglike, _ = pu.load_or_filter(
    lambda: _veber_bb_filter(df_amines_cheap, VEBER_BB["amines"]),
    accepted_csv=pu.stage_path("Amines"),
    rejected_csv=pu.rejected_path("Amines"),
    force_recompute=FORCE_RECOMPUTE,
)

pu.report_df_size(df_aldehydes_druglike,   "Aldehydes - Druglike")
pu.report_df_size(df_carboxylics_druglike, "Carboxylics - Druglike")
pu.report_df_size(df_amines_druglike,      "Amines - Druglike")

[load_or_filter] Computing Aldehydes.csv...
[Veber] Processing 36 molecules in 1 chunks...
[filter_Veber] 36/36 accepted (100.0%), 0 rejected
[filter_Veber] Cache: 0 hits, 72 misses (0.0% hit rate)
[load_or_filter] Saved Aldehydes.csv (36 accepted) + Aldehydes_rejected.csv (0 rejected)
[load_or_filter] Computing Carboxylics.csv...
[Veber] Processing 146 molecules in 1 chunks...
[filter_Veber] 145/146 accepted (99.3%), 1 rejected
[filter_Veber] Cache: 0 hits, 292 misses (0.0% hit rate)
[load_or_filter] Saved Carboxylics.csv (145 accepted) + Carboxylics_rejected.csv (1 rejected)
[load_or_filter] Computing Amines.csv...
[Veber] Processing 203 molecules in 1 chunks...
[filter_Veber] 200/203 accepted (98.5%), 3 rejected
[filter_Veber] Cache: 0 hits, 406 misses (0.0% hit rate)
[load_or_filter] Saved Amines.csv (200 accepted) + Amines_rejected.csv (3 rejected)
[Aldehydes - Druglike] 36 rows
[Carboxylics - Druglike] 145 rows
[Amines - Druglike] 200 rows


## 🔹 4. Erlenmeyer-Plöchl Reaction

Aldehydes + Carboxylic Acids → Oxazolones

In [5]:
df_oxazolones_raw = pu.load_or_run(
    lambda checkpoint_csv: pu.rxn_ErlenmeyerPlochl(df_aldehydes_druglike, df_carboxylics_druglike, checkpoint_csv=checkpoint_csv),
    output_csv=pu.stage_path("Oxazolones"),
    checkpoint_csv=pu.checkpoint_path("Oxazolones"),
    force_recompute=FORCE_RECOMPUTE,
)

pu.report_df_size(df_oxazolones_raw, "Oxazolones - Raw")

[load_or_run] Computing Oxazolones.csv...
[ErlenmeyerPlochl] 5,220 total pairs (1 chunks of ~50 aldehydes)
[ErlenmeyerPlochl] Cache: 0 hits, 5220 misses (0.0% hit rate)
[ErlenmeyerPlochl] Stats: {'input_aldehydes': 36, 'input_carboxylics': 145, 'invalid_aldehyde': 0, 'invalid_carboxyl': 0, 'not_aldehyde': 0, 'not_carboxyl': 0, 'no_product': 0, 'problematic': 0, 'output_rows': 5220, 'cache_hits': 0, 'cache_misses': 5220}
[load_or_run] Saving Oxazolones.csv (5,220 rows)
[Oxazolones - Raw] 5220 rows


## 🔸 5. Veber Filter (Oxazolones)

Apply bioavailability criteria to oxazolone intermediates.
Limits are set to the most permissive value across both downstream pathways (A-G and S-E)
to avoid discarding valid precursors prematurely.

In [6]:
VEBER_OXAZOLONES = dict(
    max_tPSA=145.99, max_RotB=10, max_LogP=5.0848,
    max_MWT=486.957, max_HBD=5, max_HBA=11
)

df_oxazolones_druglike, _ = pu.load_or_filter(
    lambda: pu.filter_Veber(pu.add_rdkit_properties(df_oxazolones_raw), **VEBER_OXAZOLONES),
    accepted_csv=pu.stage_path("Oxazolones"),
    rejected_csv=pu.rejected_path("Oxazolones"),
    force_recompute=FORCE_RECOMPUTE,
)

pu.report_df_size(df_oxazolones_druglike, "Oxazolones - Druglike")

[load_or_filter] Computing Oxazolones.csv...
[Veber] Processing 5,220 molecules in 1 chunks...
[filter_Veber] 4,087/5,220 accepted (78.3%), 1,133 rejected
[filter_Veber] Cache: 0 hits, 10440 misses (0.0% hit rate)
[load_or_filter] Saved Oxazolones.csv (4,087 accepted) + Oxazolones_rejected.csv (1,133 rejected)
[Oxazolones - Druglike] 4087 rows


## 🔹 6. Aminolysis-GFPc Reaction

Oxazolones + Amines → Imidazolones

> ⚠️ This reaction generates billions of products. Checkpointing is enabled by default.

In [7]:
df_imidazolones_raw = pu.load_or_run(
    lambda checkpoint_csv: pu.rxn_AminolysisGFPc(df_oxazolones_druglike, df_amines_druglike, checkpoint_csv=checkpoint_csv),
    output_csv=pu.stage_path("Imidazolones"),
    checkpoint_csv=pu.checkpoint_path("Imidazolones"),
    force_recompute=FORCE_RECOMPUTE,
)

pu.report_df_size(df_imidazolones_raw, "Imidazolones - Raw")

[load_or_run] Computing Imidazolones.csv...
[AminolysisGFPc] 817,400 total pairs (9 chunks of ~500 oxazolones)
[AminolysisGFPc] Cache saved to: mol_files/3. Oxazolones/.cache/aminolysis_gfpc_cache.json.gz
[AminolysisGFPc] Cache: 0 hits, 817400 misses (0.0% hit rate)
[AminolysisGFPc] ⚠️  Low cache hit rate (0.0%) - consider disabling cache for this run
[AminolysisGFPc] Stats: {'input_oxazolones': 4087, 'input_amines': 200, 'invalid_oxazolone': 0, 'invalid_amine': 0, 'not_oxazolone': 0, 'not_amine': 0, 'no_product': 0, 'problematic': 0, 'output_rows': 858270, 'cache_hits': 0, 'cache_misses': 817400}
[load_or_run] Saving Imidazolones.csv (858,270 rows)
[Imidazolones - Raw] 858270 rows


## 🔹 7. Sulphur-Exchange Reaction

Oxazolones → Thiazolones  

> **NB:** input is `df_oxazolones_druglike` (post-Veber, Cell 5) — not `df_imidazolones_raw` or `df_oxazolones_raw`.

In [8]:
THIOACETIC_PRICE_EQ = 10.0

df_thiazolones_raw = pu.load_or_run(
    lambda checkpoint_csv: pu.rxn_SulphurExchange(df_oxazolones_druglike, thioacetic_price_eq=THIOACETIC_PRICE_EQ, checkpoint_csv=checkpoint_csv),
    output_csv=pu.stage_path("Thiazolones"),
    checkpoint_csv=pu.checkpoint_path("Thiazolones"),
    force_recompute=FORCE_RECOMPUTE,
)

pu.report_df_size(df_thiazolones_raw, "Thiazolones - Raw")

[load_or_run] Computing Thiazolones.csv...
[SulphurExchange] 4,087 valid oxazolones: 0 cache hits, 4,087 misses
[SulphurExchange] Cache: 0 hits, 4087 misses (0.0% hit rate)
[SulphurExchange] Stats: {'input_oxazolones': 4087, 'thioacetic_price_eq': 10.0, 'invalid_oxazolone': 0, 'not_oxazolone': 0, 'no_product': 0, 'problematic': 0, 'output_rows': 4087, 'cache_hits': 0, 'cache_misses': 4087}
[load_or_run] Saving Thiazolones.csv (4,087 rows)
[Thiazolones - Raw] 4087 rows


## 🔸 8. Veber Filter (Products)

Compute descriptors and apply Veber criteria to both product families.  

`VEBER_PRODUCTS` defines the final target thresholds shared by imidazolones and thiazolones.

In [9]:
VEBER_PRODUCTS = dict(
    max_tPSA=140, max_RotB=10, max_LogP=5,
    max_MWT=500, max_HBD=5, max_HBA=10
)

df_imidazolones_druglike, _ = pu.load_or_filter(
    lambda: pu.filter_Veber(pu.add_rdkit_properties(df_imidazolones_raw), **VEBER_PRODUCTS),
    accepted_csv=pu.stage_path("Imidazolones"),
    rejected_csv=pu.rejected_path("Imidazolones"),
    force_recompute=FORCE_RECOMPUTE,
)

df_thiazolones_druglike, _ = pu.load_or_filter(
    lambda: pu.filter_Veber(pu.add_rdkit_properties(df_thiazolones_raw), **VEBER_PRODUCTS),
    accepted_csv=pu.stage_path("Thiazolones"),
    rejected_csv=pu.rejected_path("Thiazolones"),
    force_recompute=FORCE_RECOMPUTE,
)

pu.report_df_size(df_imidazolones_druglike, "Imidazolones - Druglike")
pu.report_df_size(df_thiazolones_druglike, "Thiazolones - Druglike")

[load_or_filter] Computing Imidazolones.csv...
[Veber] Processing 858,270 molecules in 86 chunks...
[filter_Veber] 162,291/858,270 accepted (18.9%), 695,979 rejected
[filter_Veber] Cache: 0 hits, 1716540 misses (0.0% hit rate)
[load_or_filter] Saved Imidazolones.csv (162,291 accepted) + Imidazolones_rejected.csv (695,979 rejected)
[load_or_filter] Computing Thiazolones.csv...
[Veber] Processing 4,087 molecules in 1 chunks...
[filter_Veber] 2,755/4,087 accepted (67.4%), 1,332 rejected
[filter_Veber] Cache: 0 hits, 8174 misses (0.0% hit rate)
[load_or_filter] Saved Thiazolones.csv (2,755 accepted) + Thiazolones_rejected.csv (1,332 rejected)
[Imidazolones - Druglike] 162291 rows
[Thiazolones - Druglike] 2755 rows


## 🔸 9. Brenk + PAINS Filter

Remove compounds containing Brenk structural alerts or PAINS substructures from products.

In [10]:
df_imidazolones_untoxic, _ = pu.load_or_filter(
    lambda: pu.filter_BrenkPAINS(df_imidazolones_druglike),
    accepted_csv=pu.stage_path("Imidazolones"),
    rejected_csv=pu.rejected_path("Imidazolones", "brenkpains"),
    force_recompute=FORCE_RECOMPUTE,
)

df_thiazolones_untoxic, _ = pu.load_or_filter(
    lambda: pu.filter_BrenkPAINS(df_thiazolones_druglike),
    accepted_csv=pu.stage_path("Thiazolones"),
    rejected_csv=pu.rejected_path("Thiazolones", "brenkpains"),
    force_recompute=FORCE_RECOMPUTE,
)

pu.report_df_size(df_imidazolones_untoxic, "Imidazolones - Untoxic")
pu.report_df_size(df_thiazolones_untoxic, "Thiazolones - Untoxic")

[load_or_filter] Computing Imidazolones.csv...
[filter_BrenkPAINS] 118151/162291 accepted (72.8%)
[filter_BrenkPAINS] Rejected: Brenk=38163, PAINS=5977
[load_or_filter] Saved Imidazolones.csv (118,151 accepted) + Imidazolones_brenkpains_rejected.csv (44,140 rejected)
[load_or_filter] Computing Thiazolones.csv...
[filter_BrenkPAINS] 2105/2755 accepted (76.4%)
[filter_BrenkPAINS] Rejected: Brenk=483, PAINS=167
[load_or_filter] Saved Thiazolones.csv (2,105 accepted) + Thiazolones_brenkpains_rejected.csv (650 rejected)
[Imidazolones - Untoxic] 118151 rows
[Thiazolones - Untoxic] 2105 rows


## 📤 10. Export

Imidazolones and Thiazolones saved as **separate** CSV files.

In [11]:
PROD_COLS = ["ID", "SMILES", "PriceMol", "tPSA", "RotB", "LogP", "MW", "HBD", "HBA", "MR", "HvyAtm", "Rings", "CAtm", "HetAtm"]

pu.save_dataframe(df_imidazolones_untoxic[PROD_COLS], pu.stage_path("Imidazolones"))
pu.save_dataframe(df_thiazolones_untoxic[PROD_COLS], pu.stage_path("Thiazolones"))

print("\n=== Pipeline Complete ===")
print(f"Imidazolones: {len(df_imidazolones_untoxic):,} compounds")
print(f"Thiazolones:  {len(df_thiazolones_untoxic):,} compounds")

[save_dataframe] Saved Imidazolones.csv (118,151 rows)
[save_dataframe] Saved Thiazolones.csv (2,105 rows)

=== Pipeline Complete ===
Imidazolones: 118,151 compounds
Thiazolones:  2,105 compounds
